# booth-gpu — GPU vs CPU benchmark (Colab, free T4)

Re-expresses the booth-detection hot path (color mask + connected-components + dedup/merge) as **pure GPU tensor ops** and measures the real speedup against the production CPU code.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

Then `Runtime → Run all`. The notebook clones the repo, installs deps, builds fixtures, runs the benchmark, and runs the parity check.

In [ ]:
# 0. Confirm we actually have a GPU
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU. Runtime > Change runtime type > GPU (T4), then Run all again.')
!nvidia-smi -L 2>/dev/null || true

In [ ]:
# 1. Clone the repo (edit REPO_URL to your fork/repo)
import os
REPO_URL = 'https://github.com/<you>/booth-gpu.git'   # <-- set this
if not os.path.isdir('booth-gpu'):
    !git clone $REPO_URL
%cd booth-gpu
!ls -1 src

In [ ]:
# 2. Deps. Colab already ships a CUDA torch; we only need headless OpenCV
#    for the CPU reference (cv2.inRange / connectedComponents / intersectConvexConvex).
!pip -q install opencv-python-headless

In [ ]:
# 3. Build fixtures (synthetic dense plan: ~600 true cells -> ~2k box pool)
!python src/make_fixtures.py --n-cells 600

In [ ]:
# 4. THE BENCHMARK: CPU vs GPU, with parity, for merge + mask/components
!python src/bench.py --n-cells 600

In [ ]:
# 5. Parity check (GPU output == production CPU output within tolerance)
!python tests/test_parity.py

## Use the REAL pre-NMS pool (optional)

The synthetic pool mimics a dense plan's structure. To benchmark on the *actual* box pool your pipeline produces:

1. Inside the container, copy `src/export_real_fixtures.py` to `/app`, run it in front of one dense-plan detection (see the file's docstring), and copy the resulting `boxes_pool_real.json` into `fixtures/` here.
2. Then:

```python
!python src/bench.py --pool fixtures/boxes_pool_real.json
```

The fixture is bbox+score only — no source pixels, no secrets.

In [ ]:
# 6. (optional) bigger pool to see how the GPU gap widens with N
!python src/bench.py --n-cells 1200 --no-image